<a href="https://colab.research.google.com/github/suyogbastakoti/Ai-and-Ml-Module/blob/main/2407093_SuyogBastakoti_2025_W08_Text_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Name: Suyog Bastakoti**

**Uni Id: 2407093**

# Part -1 (Class Work) : Text Pre-processing in NLP

# Basics of Text Data Cleaning

## Instructions and Requirements

In this notebook we evaluate **basic text data cleaning techniques** that are must-have for any NLP task.

We use the **NLTK** and **Regex** libraries heavily.

**Dataset:** `trumptweets_small.csv`

### To-Do:
- **Do-1:** Read and understand the provided code, then **complete Exercise-1** at the bottom.
- **Do-2:** Demonstrate the importance of text pre-processing in NLP.

---

## Key Terminology

| Term | Meaning |
|------|---------|
| **Document** | A distinct unit of text (sentence, paragraph, article) |
| **Corpus** | A collection of documents |

**Example:**
```
doc1 = "How are you?"
doc2 = "I go to school."
corpus = [doc1, doc2]
```

---


In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 1: Import Required Libraries

In [24]:
# ── Core data-handling libraries ──────────────────────────────────────────────
import pandas as pd    # For loading and working with tabular data (CSV files)
import numpy as np     # For numerical operations
import re              # Built-in Python library for Regular Expressions (Regex)


## Step 2: Load the Dataset

In [4]:
# ── Read the CSV file into a pandas DataFrame ─────────────────────────────────

df = pd.read_csv("/content/drive/MyDrive/Data Set /trumptweets_small.csv")

# Quick look at the first 5 rows
df.head()


,id,link,content,date,retweets,favorites,mentions,hashtags,geo
0,1698308935,https://twitter.com/realDonaldTrump/status/169...,Be sure to tune in and watch Donald Trump on L...,2009-05-04 20:54:25,500,868,NaN,NaN,NaN
1,1701461182,https://twitter.com/realDonaldTrump/status/170...,Donald Trump will be appearing on The View tom...,2009-05-05 03:00:10,33,273,NaN,NaN,NaN
2,1737479987,https://twitter.com/realDonaldTrump/status/173...,Donald Trump reads Top Ten Financial Tips on L...,2009-05-08 15:38:08,12,18,NaN,NaN,NaN
3,1741160716,https://twitter.com/realDonaldTrump/status/174...,New Blog Post: Celebrity Apprentice Finale and...,2009-05-08 22:40:15,11,24,NaN,NaN,NaN
4,1773561338,https://twitter.com/realDonaldTrump/status/177...,"""My persona will never be that of a wallflower...",2009-05-12 16:07:28,1399,1965,NaN,NaN,NaN


In [5]:
# ── Keep only the 'content' column (the tweet text) ──────────────────────────
# df[['content']] returns a DataFrame [Note: double brackets]
df_text = df[['content']]

# Drop rows that have missing (NaN) tweet text
df_text = df_text.dropna()

# Reset the index so it runs 0, 1, 2, ... again after dropping rows
df_text = df_text.reset_index(drop=True)

print(f"Total tweets loaded: {len(df_text)}")
df_text.head()


Total tweets loaded: 41122


,content
0,Be sure to tune in and watch Donald Trump on L...
1,Donald Trump will be appearing on The View tom...
2,Donald Trump reads Top Ten Financial Tips on L...
3,New Blog Post: Celebrity Apprentice Finale and...
4,"""My persona will never be that of a wallflower..."


## Step 3: Remove Unwanted Text

### 3a. Remove URLs

Tweets often contain links (`http://…` or `www.…`).  
We use a **regex pattern** to find and delete them.

**Regex breakdown for URLs:**
| Part | Meaning |
|------|---------|
| `https?://` | Matches `http://` **or** `https://` |
| `\S+` | One or more non-space characters |
| `\|` | OR |
| `www\.\S+` | Anything starting with `www.` |


In [6]:
def remove_urls(text):
    """
    Removes URLs from a string.

    Parameters
    ----------
    text : str  – The raw tweet text that may contain URLs.

    Returns
    -------
    str  – The same text with all URLs replaced by an empty string.
    """
    # Compile the pattern once (faster if called many times)
    url_pattern = re.compile(r'https?://\S+|www\.\S+')

    # sub() = substitute  →  replace every match with '' (nothing)
    return url_pattern.sub('', text)


# ── Quick test ────────────────────────────────────────────────────────────────
sample_url_text = "Hello! Visit us at https://www.example.com for more info."
print("Before:", sample_url_text)
print("After :", remove_urls(sample_url_text))


Before: Hello! Visit us at https://www.example.com for more info.
After : Hello! Visit us at  for more info.


In [7]:
# ── Apply to the entire column ────────────────────────────────────────────────
# .apply() runs the function on EVERY row in the column
text_no_url = df_text["content"].apply(remove_urls)

# Show first 3 results
text_no_url.head(3)


,content
0,Be sure to tune in and watch Donald Trump on L...
1,Donald Trump will be appearing on The View tom...
2,Donald Trump reads Top Ten Financial Tips on L...


### 3b. Remove Emojis

Emojis are stored as Unicode characters (e.g. 😊 = U+1F60A).  
We build a pattern that covers the most common emoji Unicode ranges.


In [8]:
def remove_emoji(text):
    """
    Replaces emoji characters with a single space.

    Parameters
    ----------
    text : str  – Text that may contain emoji.

    Returns
    -------
    str  – Text with emoji replaced by spaces.
    """
    # Each Unicode range covers a category of emoji / symbols
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"  # emoticons  😀 😭 😂 …
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs  🌍 🎃 🔥 …
        u"\U0001F680-\U0001F6FF"  # transport & map  🚀 🚗 ✈️ …
        u"\U0001F1E0-\U0001F1FF"  # flags  🇺🇸 🇬🇧 …
        u"\U00002702-\U000027B0"  # dingbats  ✂ ✅ …
        u"\U000024C2-\U0001F251"  # enclosed characters  ℹ Ⓜ …
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub(' ', text)   # Replace emoji with a space


# ── Quick test ────────────────────────────────────────────────────────────────
test_emoji = "Hello @siman 👋🏾, still on for the movie??? #MovieNight 🍿"
print("Before:", test_emoji)
print("After :", remove_emoji(test_emoji))


Before: Hello @siman 👋🏾, still on for the movie??? #MovieNight 🍿
After : Hello @siman  , still on for the movie??? #MovieNight  


### 3c. Remove All Unwanted Characters

We combine several cleaning steps into **one function**:
1. Remove `@mentions`
2. Remove `#hashtags`
3. Remove punctuation / special characters
4. Remove emoji
5. Remove double spaces
6. Strip leading / trailing whitespace


In [9]:
def removeunwanted_characters(document):
    """
    Cleans a single tweet by removing mentions, hashtags, punctuation, emoji,
    extra spaces, and surrounding whitespace.

    Parameters
    ----------
    document : str  – Raw tweet text.

    Returns
    -------
    str  – Cleaned tweet text.
    """
    # 1.Remove @mentions  (e.g. @realDonaldTrump)
    #    Pattern: @ followed by letters, digits, or underscores
    document = re.sub(r'@[A-Za-z0-9_]+', ' ', document)

    # 2. Remove #hashtags  (e.g. #MAGA)
    document = re.sub(r'#[A-Za-z0-9_]+', '', document)

    # 3. Remove anything that is NOT a letter, digit, or space
    #    (strips punctuation, special symbols, etc.)
    document = re.sub(r'[^0-9A-Za-z ]', '', document)

    # 4. Remove emoji
    document = remove_emoji(document)

    # 5. Collapse double spaces down to single space
    document = document.replace('  ', ' ')

    # 6. Remove leading / trailing whitespace
    return document.strip()


# ── Quick test ────────────────────────────────────────────────────────────────
sample_tweet = "Hello @gabe_flomo 👋🏾, still want to hit that sushi spot??? #sushiBros #rawFish #🍱"
print("Before:", sample_tweet)
print("After :", removeunwanted_characters(sample_tweet))


Before: Hello @gabe_flomo 👋🏾, still want to hit that sushi spot??? #sushiBros #rawFish #🍱
After : Hello  still want to hit that sushi spot


In [10]:
# ── Apply to the full dataset ─────────────────────────────────────────────────
text_removed_unwanted = df_text["content"].apply(removeunwanted_characters)
text_removed_unwanted.head(5)


,content
0,Be sure to tune in and watch Donald Trump on L...
1,Donald Trump will be appearing on The View tom...
2,Donald Trump reads Top Ten Financial Tips on L...
3,New Blog Post Celebrity Apprentice Finale and ...
4,My persona will never be that of a wallflower ...


## Step 4: Tokenisation

**Tokenisation** = splitting a sentence into individual words (tokens).

**Example:**
```
IN : "He did not try to navigate."
OUT: ['He', 'did', 'not', 'try', 'to', 'navigate', '.']
```

We use NLTK's `word_tokenize` which handles contractions and punctuation correctly.


In [11]:
import nltk

# Download the 'punkt_tab' tokeniser rules (only needed once)
nltk.download('punkt_tab', quiet=True)

from nltk import word_tokenize   # The actual tokeniser function


# ── Quick test ────────────────────────────────────────────────────────────────
sample_sentence = "He did not try to navigate after the first bold flight, for the reaction had taken something out of his soul."
tokens = word_tokenize(sample_sentence)
print("Tokens:", tokens)
print(f"Total tokens: {len(tokens)}")


Tokens: ['He', 'did', 'not', 'try', 'to', 'navigate', 'after', 'the', 'first', 'bold', 'flight', ',', 'for', 'the', 'reaction', 'had', 'taken', 'something', 'out', 'of', 'his', 'soul', '.']
Total tokens: 23


### Remove Punctuation Using RegexpTokenizer

`RegexpTokenizer(r'\w+')` keeps only **word characters** (letters, digits, `_`).  
All punctuation is automatically discarded during tokenisation.


In [12]:
from nltk.tokenize import RegexpTokenizer


def remove_punct(text_tokens):
    """
    Takes a list of tokens (which may still have punctuation marks as tokens)
    and returns only the 'word' tokens — removing standalone punctuation.

    Parameters
    ----------
    text_tokens : list  – List of string tokens.

    Returns
    -------
    list  – Clean list of word tokens (no punctuation tokens).
    """
    # Join the tokens back into a string first, then re-tokenise
    # using \w+ (word characters only) — punctuation tokens are ignored
    tokenizer = RegexpTokenizer(r'\w+')
    return tokenizer.tokenize(' '.join(text_tokens))


# ── Quick test ────────────────────────────────────────────────────────────────
messy_text = "He did not try: after the!!!! first bold flight, for,,,,, the reaction!!!!had taken??????? something out of his soul."
step1 = word_tokenize(messy_text)        # Step 1: tokenise (includes punctuation marks)
step2 = remove_punct(step1)             # Step 2: drop punctuation tokens

print("Original      :", messy_text)
print("After tokenise:", step1)
print("After remove  :", step2)


Original      : He did not try: after the!!!! first bold flight, for,,,,, the reaction!!!!had taken??????? something out of his soul.
After tokenise: ['He', 'did', 'not', 'try', ':', 'after', 'the', '!', '!', '!', '!', 'first', 'bold', 'flight', ',', 'for', ',', ',', ',', ',', ',', 'the', 'reaction', '!', '!', '!', '!', 'had', 'taken', '?', '?', '?', '?', '?', '?', '?', 'something', 'out', 'of', 'his', 'soul', '.']
After remove  : ['He', 'did', 'not', 'try', 'after', 'the', 'first', 'bold', 'flight', 'for', 'the', 'reaction', 'had', 'taken', 'something', 'out', 'of', 'his', 'soul']


## Step 5: Remove Stop Words

**Stop words** are common words that carry little meaning on their own:  
`"the"`, `"and"`, `"is"`, `"in"`, `"a"`, `"an"`, …

We use NLTK's built-in English stop-word list and **add custom ones** for Twitter data.


In [13]:
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

# Load the English stop-word list into a set (fast O(1) look-up)
stop_words = set(stopwords.words('english'))

# Add Twitter-specific tokens that are noise in our dataset
custom_stopwords = ['@', 'RT', 'http', 'https']
stop_words.update(custom_stopwords)

print(f"Total stop words (incl. custom): {len(stop_words)}")
print("Sample:", list(stop_words)[:10])


Total stop words (incl. custom): 202
Sample: ['m', 'o', 'doesn', 'be', "hadn't", 'been', "i've", 'hadn', 'those', 'an']


In [14]:
def remove_stopwords(text_tokens):
    """
    Removes stop words from a list of tokens.

    Parameters
    ----------
    text_tokens : list  – Tokenised input text.

    Returns
    -------
    list  – Tokens with stop words filtered out.
    """
    # Keep only tokens that are NOT in our stop-word set
    return [token for token in text_tokens if token not in stop_words]


# ── Quick test ────────────────────────────────────────────────────────────────
sample_tokens = ['He', 'did', 'not', 'try', 'to', 'navigate', 'after', 'the',
                 'first', 'bold', 'flight', 'for', 'the', 'reaction']

print("Before:", sample_tokens)
print("After :", remove_stopwords(sample_tokens))


Before: ['He', 'did', 'not', 'try', 'to', 'navigate', 'after', 'the', 'first', 'bold', 'flight', 'for', 'the', 'reaction']
After : ['He', 'try', 'navigate', 'first', 'bold', 'flight', 'reaction']


## Step 6: Text Normalisation

**Goal:** Reduce the vocabulary size by mapping different forms of the same word to a single root.

Two main techniques:

| Technique | Method | Example |
|-----------|--------|---------|
| **Lemmatisation** | Uses grammar rules → proper word form | running → run |
| **Stemming** | Chops off suffixes → may not be a real word | running → run, connections → connect |

### 6a. Lemmatisation


In [15]:
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('wordnet', quiet=True)

from nltk.stem import WordNetLemmatizer


def lemmatization(token_text):
    """
    Lemmatises each token using the WordNet lemmatiser.

    The pos='v' argument tells the lemmatiser to treat every word as a VERB.
    This works well for tweet data where verbs dominate the action words.

    Parameters
    ----------
    token_text : list  – List of string tokens.

    Returns
    -------
    list  – List of lemmatised tokens.
    """
    wordnet_lem = WordNetLemmatizer()
    return [wordnet_lem.lemmatize(token, pos='v') for token in token_text]


# ── Quick test ────────────────────────────────────────────────────────────────
test_words = "Should we go walking or swimming".split()
print("Before:", test_words)
print("After :", lemmatization(test_words))


Before: ['Should', 'we', 'go', 'walking', 'or', 'swimming']
After : ['Should', 'we', 'go', 'walk', 'or', 'swim']


### 6b. Stemming

In [16]:
from nltk.stem import PorterStemmer


def stemming(text_tokens):
    """
    Applies Porter Stemming to each token.

    Stemming is more aggressive than lemmatisation — it simply chops off
    word endings. The result may not be a real English word.

    Parameters
    ----------
    text_tokens : list  – List of string tokens.

    Returns
    -------
    list  – List of stemmed tokens.
    """
    porter = PorterStemmer()
    return [porter.stem(word) for word in text_tokens]


# ── Compare Lemmatisation vs Stemming ─────────────────────────────────────────
test_words = ['Connects', 'Connecting', 'Connections', 'Connected', 'Connection', 'Connect']

print("Original Tokens  :", test_words)
print("Lemmatised Tokens:", lemmatization(test_words))
print("Stemmed Tokens   :", stemming(test_words))


Original Tokens  : ['Connects', 'Connecting', 'Connections', 'Connected', 'Connection', 'Connect']
Lemmatised Tokens: ['Connects', 'Connecting', 'Connections', 'Connected', 'Connection', 'Connect']
Stemmed Tokens   : ['connect', 'connect', 'connect', 'connect', 'connect', 'connect']


### 6c. Convert to Lower Case

In [17]:
def lower_order(text):
    """
    Converts all characters in a string to lower case.

    Why? So 'Trump' and 'trump' are treated as the same word by the model.

    Parameters
    ----------
    text : str  – Input text.

    Returns
    -------
    str  – Lower-cased text.
    """
    return text.lower()


# ── Quick test ────────────────────────────────────────────────────────────────
sample = "This Is Some NORMALIZED Text."
print("Before:", sample)
print("After :", lower_order(sample))


Before: This Is Some NORMALIZED Text.
After : this is some normalized text.


---

## Exercise-1: Complete the Text Cleaning Pipeline

**Goal:** Combine all the individual cleaning steps into **one pipeline function** and apply it to the `trumptweets_small.csv` dataset.

The function should:
1. Lower-case the text
2. Remove URLs
3. Remove emoji
4. Remove unwanted characters (mentions, hashtags, punctuation)
5. Tokenise the text
6. Remove stop words
7. Apply either **lemmatisation** or **stemming** (controlled by the `rule` parameter)
8. Return the cleaned tokens joined back into a string

---

### Load the dataset first


In [18]:
# ── Load the dataset ──
# 'content' column contains the raw tweet text
data = pd.read_csv("/content/drive/MyDrive/Data Set /trumptweets_small.csv")

# Keep only the tweet text column and drop empty rows
data_cleaning = data["content"].dropna().reset_index(drop=True)

print(f"Total tweets: {len(data_cleaning)}")
print("\nFirst tweet (raw):")
print(data_cleaning[0])


Total tweets: 41122

First tweet (raw):
Be sure to tune in and watch Donald Trump on Late Night with David Letterman as he presents the Top Ten List tonight!


### Completed Pipeline Function

Below is the **corrected and completed** version of the pipeline.

**Bug fixes from the original Exercise-1 template:**
- Each cleaning step now correctly chains onto the result of the previous step (not always back to `dataset`).
- Tokenisation is done with `word_tokenize` on a single string, not `.split()`, so contractions are handled properly.
- The function handles both `"lemmatize"` and `"stem"` rules, and warns for any other input.


In [19]:
def text_cleaning_pipeline(text, rule="lemmatize"):
    """
    Runs a full NLP pre-processing pipeline on a single tweet string.

    Steps (in order):
        1. Lower-case  →  all characters become small letters
        2. Remove URLs  →  http/https/www links deleted
        3. Remove emoji  →  Unicode emoji replaced with space
        4. Remove unwanted chars  →  mentions, hashtags, punctuation removed
        5. Tokenise  →  string split into list of word tokens
        6. Remove stop words  →  common noise words discarded
        7. Normalise  →  lemmatise OR stem every remaining token
        8. Re-join  →  list of tokens joined back into one string

    Parameters
    ----------
    text : str
        A single raw tweet string.
    rule : str, optional
        'lemmatize' (default) – use WordNet lemmatiser
        'stem'                – use Porter stemmer

    Returns
    -------
    str  – The fully cleaned tweet text as a single string.
    """
    # ── Step 1: Lower-case ───────────────────────────────────────────────────
    text = lower_order(text)

    # ── Step 2: Remove URLs ──────────────────────────────────────────────────
    text = remove_urls(text)

    # ── Step 3: Remove emoji ─────────────────────────────────────────────────
    text = remove_emoji(text)

    # ── Step 4: Remove mentions, hashtags, punctuation, etc. ─────────────────
    text = removeunwanted_characters(text)

    # ── Step 5: Tokenise (split string → list of word tokens) ────────────────
    tokens = word_tokenize(text)

    # ── Step 6: Remove stop words ────────────────────────────────────────────
    tokens = remove_stopwords(tokens)

    # ── Step 7: Normalise ────────────────────────────────────────────────────
    if rule == "lemmatize":
        wordnet_lem = WordNetLemmatizer()
        tokens = [wordnet_lem.lemmatize(token, pos='v') for token in tokens]
    elif rule == "stem":
        porter = PorterStemmer()
        tokens = [porter.stem(token) for token in tokens]
    else:
        print(f"Unknown rule '{rule}'. Choose 'lemmatize' or 'stem'.")

    # ── Step 8: Re-join tokens into a string ─────────────────────────────────
    return " ".join(tokens)


# ─────────────────────────────────────────────────────────────────────────────
# Quick test with a sample tweet
# ─────────────────────────────────────────────────────────────────────────────
sample = ("Hello @gabe_flomo, I still want us to hit that new sushi spot??? "
          "LMK when you're free cuz I can't go this or next weekend! "
          "https://example.com #sushiBros #rawFish")

print("Raw tweet:")
print(sample)
print("\nAfter pipeline (lemmatize):")
print(text_cleaning_pipeline(sample, rule="lemmatize"))
print("\nAfter pipeline (stem):")
print(text_cleaning_pipeline(sample, rule="stem"))


Raw tweet:
Hello @gabe_flomo, I still want us to hit that new sushi spot??? LMK when you're free cuz I can't go this or next weekend! https://example.com #sushiBros #rawFish

After pipeline (lemmatize):
hello still want us hit new sushi spot lmk youre free cuz cant go next weekend

After pipeline (stem):
hello still want us hit new sushi spot lmk your free cuz cant go next weekend


In [20]:
# ── Test on the first real tweet from the dataset ─────────────────────────────
raw_tweet = data_cleaning[0]
print("Raw tweet:")
print(raw_tweet)
print("\nCleaned (lemmatize):")
print(text_cleaning_pipeline(raw_tweet, rule="lemmatize"))


Raw tweet:
Be sure to tune in and watch Donald Trump on Late Night with David Letterman as he presents the Top Ten List tonight!

Cleaned (lemmatize):
sure tune watch donald trump late night david letterman present top ten list tonight


In [21]:
# ── Apply the pipeline to the ENTIRE dataset ─────────────────────────────────
# lambda wraps the function so .apply() can pass each tweet to it
cleaned_tokens = data_cleaning.apply(
    lambda tweet: text_cleaning_pipeline(tweet, rule="lemmatize")
)

print(f"Total cleaned tweets: {len(cleaned_tokens)}")
print("\nFirst 5 cleaned tweets:")
cleaned_tokens.head()


Total cleaned tweets: 41122

First 5 cleaned tweets:


,content
0,sure tune watch donald trump late night david ...
1,donald trump appear view tomorrow morning disc...
2,donald trump read top ten financial tip late s...
3,new blog post celebrity apprentice finale less...
4,persona never wallflower id rather build wall ...


In [22]:
# ── Side-by-side comparison: raw vs. cleaned ─────────────────────────────────
comparison = pd.DataFrame({
    "raw_tweet"    : data_cleaning,
    "cleaned_tweet": cleaned_tokens
})

# Show 3 random examples
comparison.sample(3, random_state=42).reset_index(drop=True)


,raw_tweet,cleaned_tweet
0,Great ruling on wind farm in Scotland—very sma...,great rule wind farm scotlandvery smart judge ...
1,# MakeAmericaGreatAgain # Trump2016 LIFE CHANG...,makeamericagreatagain trump2016 life change ex...
2,""" @ ellenEspence: I'm not convinced that any c...",ellenespence im convince candidate realdonaldt...


---

## Summary — What We Did

| Step | Function | Purpose |
|------|----------|---------|
| 1 | `lower_order()` | Standardise case |
| 2 | `remove_urls()` | Remove web links |
| 3 | `remove_emoji()` | Remove emoji Unicode |
| 4 | `removeunwanted_characters()` | Remove @mentions, #hashtags, punctuation |
| 5 | `word_tokenize()` | Split text into word tokens |
| 6 | `remove_stopwords()` | Filter common noise words |
| 7 | `lemmatization()` / `stemming()` | Reduce words to root form |

**Why does this matter?**
- Raw tweet text is extremely noisy (URLs, emoji, slang, mentions).
- NLP models perform far better on clean, normalised text.
- This pipeline converts messy social-media data into structured token sequences ready for downstream tasks like sentiment analysis or topic modelling.
